# QuantJourney SDK - Earnings Revisions and Price Momentum

This notebook demonstrates a QuantJourney SDK workflow that ranks forward EPS and revenue revisions, breadth, dispersion, acceleration and price momentum across a peer set.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Market Data Helpers

In [ ]:
def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


In [ ]:
symbols = ['NVDA', 'MSFT', 'AAPL', 'AMZN', 'GOOGL', 'META', 'AVGO', 'AMD']
estimates_raw = {symbol: qj.fmp.get_analyst_estimates(symbol=symbol, period='annual', limit=12) for symbol in symbols}
surprises_raw = {symbol: qj.fmp.get_earnings_surprises(symbol=symbol) for symbol in symbols}
targets_raw = {symbol: qj.fmp.get_price_target_summary(symbol=symbol) for symbol in symbols}
ratios_raw = {symbol: qj.fmp.get_financial_ratios_ttm(symbol=symbol) for symbol in symbols}
prices, volumes = price_panel(symbols, start='2023-01-01', end=END)


In [ ]:
rows = []
for symbol in symbols:
    estimates = pd.DataFrame(as_rows(estimates_raw[symbol]))
    numeric = estimates.select_dtypes(include='number')
    eps_path = numeric.mean(axis=1).head(3).to_numpy() if not numeric.empty else np.array([])
    if len(eps_path) < 3:
        eps_path = np.array([np.nan, np.nan, np.nan])
    surprise_rows = pd.DataFrame(as_rows(surprises_raw[symbol]))
    target_rows = pd.DataFrame(as_rows(targets_raw[symbol]))
    ratio_row = pd.DataFrame(as_rows(ratios_raw[symbol])).head(1)
    rows.append({'symbol': symbol, 'cfy_estimate': eps_path[0], 'nfy_estimate': eps_path[1], 'fy2_estimate': eps_path[2], 'revision_breadth': numeric.diff().gt(0).mean().mean() if not numeric.empty else np.nan, 'revision_dispersion': numeric.std(axis=1).mean() / numeric.mean(axis=1).abs().mean() if not numeric.empty else np.nan, 'surprise_rows': len(surprise_rows), 'price_target_rows': len(target_rows), 'pe_ttm': pd.to_numeric(ratio_row.get('peRatioTTM', pd.Series([np.nan])).iloc[0], errors='coerce') if not ratio_row.empty else np.nan})
revisions = pd.DataFrame(rows).set_index('symbol')


In [ ]:
momentum_12_1 = prices.pct_change(252).iloc[-1] - prices.pct_change(21).iloc[-1]
momentum_3m = prices.pct_change(63).iloc[-1]
revisions['eps_acceleration'] = revisions['fy2_estimate'] - revisions['cfy_estimate']
revisions['price_momentum_12_1'] = momentum_12_1.reindex(revisions.index)
revisions['price_momentum_3m'] = momentum_3m.reindex(revisions.index)
revisions['composite_revision_momentum'] = revisions['revision_breadth'].rank(pct=True) - revisions['revision_dispersion'].rank(pct=True) + revisions['eps_acceleration'].rank(pct=True) + revisions['price_momentum_12_1'].rank(pct=True)
display(revisions.sort_values('composite_revision_momentum', ascending=False))
revisions[['revision_breadth', 'revision_dispersion', 'eps_acceleration', 'price_momentum_12_1']].plot(kind='bar', subplots=True, layout=(2, 2), figsize=(14, 7), title='Revisions and momentum inputs')
plt.tight_layout()
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.